# Modeling Phase

In this section, we develop and evaluate machine learning models to address the problem statement. We will explore different algorithms, tune hyperparameters, and assess model performance using appropriate metrics.

## 1. Feature Scaling

Before modeling, we scale the selected features to ensure all variables contribute equally to the analysis. This step helps improve model performance and convergence, especially for algorithms sensitive to feature magnitude.

In [15]:
#import libraries
import pandas as pd
import numpy as np
from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.impute import SimpleImputer

In [5]:
# Load and prepare data
df = pd.read_csv('..\Data\engineered_gk_features.csv')
df = df.dropna(subset=['Market_Value_Numeric'])

<>:2: SyntaxWarning: invalid escape sequence '\D'
<>:2: SyntaxWarning: invalid escape sequence '\D'
C:\Users\azedd\AppData\Local\Temp\ipykernel_17544\2900590585.py:2: SyntaxWarning: invalid escape sequence '\D'
  df = pd.read_csv('..\Data\engineered_gk_features.csv')


In [11]:
# 1. Pre-scaling inspection
print("Original Features Summary:")
print(df.describe().T[['mean', 'std', 'min', 'max']])

# Visualize feature distributions
plt.figure(figsize=(15, 20))
for i, col in enumerate(df.select_dtypes(include=np.number).columns, 1):
    plt.subplot(10, 6, i)
    sns.histplot(df[col], kde=True)
    plt.title(f'{col} Distribution')
    plt.tight_layout()
plt.savefig('feature_distributions_pre_scaling.png', dpi=300)
plt.close()

Original Features Summary:
                              mean           std            min           max
GA                    2.503571e+01  1.866224e+01       0.000000  6.400000e+01
Command_index         3.897030e-01  7.828962e-02       0.288187  8.000000e-01
Market_Value_Numeric  8.453214e+06  9.316791e+06  150000.000000  4.000000e+07
GA90                  1.341786e+00  5.087243e-01       0.000000  3.000000e+00
Save_consistency      9.887829e-01  2.386106e-02       0.865019  1.000000e+00
League_strength       7.946452e-01  8.582762e-02       0.651770  9.083900e-01
Shot_stopping_index   4.956701e-01  1.472089e-01       0.129870  8.490000e-01
PKatt                 2.521429e+00  2.433187e+00       0.000000  1.000000e+01
Adj_GA90              1.676349e+00  7.084089e-01       0.000000  3.835709e+00
Starts                1.893571e+01  1.359550e+01       1.000000  3.800000e+01
MP                    1.905714e+01  1.350714e+01       1.000000  3.800000e+01
Goals_prevented       8.988143e-01  5

In [12]:
# 2. Prepare data
# Separate features and target
features = df.drop(columns=['Market_Value_Numeric'], errors='ignore')
target = df['Market_Value_Numeric'] if 'Market_Value_Numeric' in df else None

In [13]:
# Identify numerical features (exclude non-numeric and target)
numerical_features = features.select_dtypes(include=np.number).columns.tolist()

In [16]:
# 3. Create robust scaling pipeline
scaling_pipeline = make_pipeline(
    SimpleImputer(strategy='median'),  # Handle missing values
    RobustScaler(quantile_range=(25, 75)) )  # Outlier-resistant scaling

In [17]:
# 4. Apply scaling
scaled_features = scaling_pipeline.fit_transform(features[numerical_features])

In [18]:
# 5. Create scaled DataFrame
scaled_df = pd.DataFrame(
    scaled_features,
    columns=[f"{col}_scaled" for col in numerical_features]
)

In [19]:
# 6. Reintegrate non-numeric columns and target
final_df = pd.concat([
    features.drop(columns=numerical_features),
    scaled_df,
    df[['Market_Value_Numeric']] if target is not None else pd.DataFrame()
], axis=1)


In [20]:
# 7. Post-scaling inspection
print("\nScaled Features Summary:")
print(scaled_df.describe().T[['mean', 'std', 'min', 'max']])

# Visualize scaled distributions
plt.figure(figsize=(15, 20))
for i, col in enumerate(scaled_df.columns, 1):
    plt.subplot(10, 6, i)
    sns.histplot(scaled_df[col], kde=True)
    plt.title(f'{col} Distribution')
    plt.tight_layout()
plt.savefig('feature_distributions_post_scaling.png', dpi=300)
plt.close()


Scaled Features Summary:
                                mean       std        min       max
GA_scaled                   0.030462  0.548889  -0.705882  1.176471
Command_index_scaled        0.084222  0.840508  -1.005645  4.489119
GA90_scaled                 0.021725  0.937741  -2.451613  3.078341
Save_consistency_scaled    -0.913526  2.878387 -15.843326  0.439608
League_strength_scaled     -0.181869  1.231673  -2.631236  1.650759
Shot_stopping_index_scaled -0.030945  0.660928  -1.673289  1.555410
PKatt_scaled                0.130357  0.608297  -0.500000  2.000000
Adj_GA90_scaled             0.085021  1.208083  -3.218663  4.410487
Starts_scaled              -0.002381  0.503537  -0.666667  0.703704
MP_scaled                   0.002116  0.500264  -0.666667  0.703704
Goals_prevented_scaled      0.127751  0.974170  -2.426992  3.027923
Saves_per_90_scaled         0.147530  0.993820  -2.175085  3.448965
Mins_per_match_scaled      -1.860922  5.232265 -36.666667  0.000000
L_scaled              

In [21]:
# 8. Save scaled dataset
final_df.to_csv('scaled_gk_features.csv', index=False)
print("Scaled dataset saved to 'scaled_gk_features.csv'")


Scaled dataset saved to 'scaled_gk_features.csv'


In [22]:
# 9. Optional: Target transformation (highly recommended)
if target is not None:
    # Apply log transformation to market values
    log_target = np.log1p(target)
    
    # Visualize transformation
    fig, ax = plt.subplots(1, 2, figsize=(12, 5))
    sns.histplot(target, ax=ax[0], kde=True)
    ax[0].set_title('Original Market Values')
    sns.histplot(log_target, ax=ax[1], kde=True)
    ax[1].set_title('Log-Transformed Market Values')
    plt.savefig('target_transformation.png', dpi=300)
    plt.close()
    
    # Add to final dataset
    final_df['Log_Market_Value'] = log_target
    final_df.to_csv('scaled_gk_features.csv', index=False)
    print("Added log-transformed target to dataset")

Added log-transformed target to dataset
